# Car Price Prediction — Data Cleaning

## Overview

This notebook fixes every problem we identified in notebook 01.

Raw data from PakWheels is messy — prices are text, mileage has commas,
engine sizes have units attached, and the make column is completely empty.

## What This Notebook Does

| Step | Problem | Fix |
|------|---------|-----|
| 1 | Price stored as "PKR 38.75 lacs" | Extract number, convert to rupees |
| 2 | Mileage stored as "19,198 km" | Remove commas and "km", convert to integer |
| 3 | Engine stored as "660 cc" | Remove "cc", convert to integer |
| 4 | Make column is empty | Extract brand name from title column |
| 5 | Unrealistic year values | Filter to 1980 — 2026 only |
| 6 | Missing values | Drop or fill based on column importance |
| 7 | Save clean data | Save to data/processed/02_cleaned.csv |

## 1. Importing Libraries

We need pandas for data manipulation and OS for file paths.
Re is a new library here — it stands for Regular Expressions.
It helps us extract numbers from messy text like "PKR 38.75 lacs".

In [ ]:
import pandas as pd
import os
import re

pandas — the main tool for working with data tables in Python.
Think of it like Excel but inside your code.

os — helps you build file paths like "go to the data folder, then raw folder".
Without it, your code might break on a different computer.

re — stands for Regular Expressions. It is a search tool for text.
Imagine you have the text "PKR 38.75 lacs" and you only want the number 38.75.
re helps you find and pull out exactly that number, ignoring everything else.

## 2. Loading the Raw Data

We always load from the raw file in the cleaning notebook.
We never load from a previously cleaned file here.
This way our cleaning steps are always reproducible from scratch.

In [ ]:
data_path = os.path.join('..', 'data', 'raw', 'pakwheels.csv')
df = pd.read_csv(data_path)
print(f"Rows loaded: {len(df)}")

Good. We have the raw data loaded and ready to clean.
We will track the row count after each step so we know
exactly how many rows each cleaning step removes.
Think of it like a scoreboard — we start with X rows
and we want to keep as many good rows as possible.

## 3. Cleaning the Price Column

Right now price looks like this: "PKR 38.75 lacs"
We need it to look like this: 3875000

Here is the plan:
- Extract the number 38.75 from the text
- Multiply by 100,000 to convert lacs to rupees
- Store it as a proper number column

In [ ]:
def clean_price(price_str):
    if pd.isnull(price_str):
        return None
    number = re.search(r'[\d.]+', str(price_str))
    return float(number.group()) * 100000 if number else None

df['price_pkr'] = df['price'].apply(clean_price)
print(df[['price', 'price_pkr']].head(3))

The price column is now a proper number in rupees.

Let us break down every single line of the code above.

`def clean_price(price_str):` — we are creating our own function.
A function is like a recipe. You give it an ingredient and it gives you back a result.
Here the ingredient is one price value like "PKR 38.75 lacs".

`if pd.isnull(price_str): return None` — before doing anything,
we check if the value is empty. If it is empty, we return None which means "no value".
This stops the code from crashing on empty rows.

`re.search(r'[\d.]+', str(price_str))` — this is the search tool.
Think of it like a metal detector scanning the text "PKR 38.75 lacs".
The metal detector is looking for digits (0-9) and dots (.).
It finds "38.75" and stops there.

`float(number.group())` — number.group() grabs the match it found, which is "38.75" as text.
float() converts that text into a real decimal number so we can do math with it.

`* 100000` — we multiply by 100,000 because 1 lac = 100,000.
So 38.75 lacs becomes 38.75 × 100,000 = 3,875,000 rupees.

`df['price_pkr'] = df['price'].apply(clean_price)` — apply() runs our function
on every single row in the price column automatically.
The result is stored in a new column called price_pkr.

## 4. Cleaning the Mileage Column

Right now mileage looks like this: "19,198 km"
We need it to look like this: 19198

Here is the plan:
- Remove the comma from "19,198"
- Remove the word "km"
- Convert what is left into a whole number

In [ ]:
def clean_mileage(mileage_str):
    if pd.isnull(mileage_str):
        return None
    number = re.search(r'[\d,]+', str(mileage_str))
    return int(number.group().replace(',', '')) if number else None

df['mileage_km'] = df['mileage'].apply(clean_mileage)
print(df[['mileage', 'mileage_km']].head(3))

The mileage column is now a clean whole number.

Let us break down every single line.

`def clean_mileage(mileage_str):` — we are creating a new function.
Just like the price function, this one takes one mileage value at a time
and returns a clean number back.

`if pd.isnull(mileage_str): return None` — we check if the value is empty first.
If it is empty we return None so the code does not crash.
Always check for empty values before doing anything else.

`re.search(r'[\d,]+', str(mileage_str))` — our metal detector is scanning the text.
This time it is looking for digits AND commas together.
So from "19,198 km" it finds "19,198".
We include the comma in the search so we can remove it in the next step.

`number.group().replace(',', '')` — number.group() grabs "19,198" as text.
replace(',', '') removes every comma from it, giving us "19198" as text.

`int(...)` — converts the text "19198" into a whole number 19198.
We use int here instead of float because mileage is always a whole number.
No one drives 19198.5 kilometres.

`df['mileage_km'] = df['mileage'].apply(clean_mileage)` — apply() runs our function
on every single row automatically and saves the result in a new column called mileage_km.

## 5. Cleaning the Engine Column

Right now engine size looks like this: "660 cc"
We need it to look like this: 660

Here is the plan:
- Find the number before "cc"
- Remove the "cc" part
- Convert what is left into a whole number

In [ ]:
def clean_engine(engine_str):
    if pd.isnull(engine_str):
        return None
    number = re.search(r'[\d,]+', str(engine_str))
    return int(number.group().replace(',', '')) if number else None

df['engine_cc'] = df['engine'].apply(clean_engine)
print(df[['engine', 'engine_cc']].head(3))

The engine column is now a clean whole number in cc.

Let us break down every single line.

`def clean_engine(engine_str):` — we create another function.
This one takes one engine value at a time like "660 cc"
and gives back just the number 660.

`if pd.isnull(engine_str): return None` — same safety check as before.
If the value is empty, return None and move on.
We always do this first before touching the value.

`re.search(r'[\d,]+', str(engine_str))` — the metal detector scans "660 cc"
and finds "660". It stops when it hits a space before "cc".

`number.group().replace(',', '')` — grabs the found text and removes any commas.
Some engine sizes like "1,300 cc" have a comma, so we clean that too.

`int(...)` — converts the text "660" into the number 660.
Engine size is always a whole number so we use int, not float.

`df['engine_cc'] = df['engine'].apply(clean_engine)` — runs the function
on every row and saves the result in a new column called engine_cc.

We now have three clean numeric columns: price_pkr, mileage_km, engine_cc.
These three were the messiest columns in the entire dataset.

## 6. Extracting the Make Column

The make column in our dataset is completely empty.
But the car brand is hidden inside the title column.

For example:
"Honda Civic 2020 VTi for Sale" — the make is "Honda"
"Toyota Corolla 2019 GLi for Sale" — the make is "Toyota"
"Suzuki Mehran 2015 for Sale" — the make is "Suzuki"

The brand is always the very first word in the title.
So we just need to grab the first word from every title.

In [ ]:
df['make'] = df['title'].str.split().str[0].str.strip()
print(df[['title', 'make']].head(5))

The make column is now filled with the brand name for every car.

Let us break down every single part of that one line.

`df['title']` — this selects the title column from our table.
Each value looks like "Honda Civic 2020 VTi for Sale".

`.str` — this tells pandas we want to work with the text inside the column.
Without .str, pandas would not know we want to treat the values as text.

`.split()` — this splits the text into a list of separate words.
"Honda Civic 2020 VTi for Sale" becomes ["Honda", "Civic", "2020", "VTi", "for", "Sale"].
Think of it like cutting a sentence into individual word pieces.

`.str[0]` — this picks the first item from every list.
From ["Honda", "Civic", "2020", "VTi", "for", "Sale"] it picks "Honda".
Remember counting starts at 0 in Python, so [0] means the first word.

`.str.strip()` — this removes any accidental spaces from the beginning or end.
Sometimes scraped data has hidden spaces that cause problems later.

`df['make'] = ...` — saves the result into the make column,
overwriting the empty values that were there before.

## 7. Filtering Unrealistic Year Values

The year column tells us when each car was manufactured.
Any year before 1980 or after 2026 is almost certainly a data error.

A car from 1960 listed on PakWheels is extremely unlikely.
A car from 2030 has not been manufactured yet.
We remove these rows to keep our data realistic.

In [ ]:
before = len(df)
df = df[(df['year'] >= 1980) & (df['year'] <= 2026)]
print(f"Rows before: {before}")
print(f"Rows after:  {len(df)}")
print(f"Removed:     {before - len(df)}")

Let us break down every single part of this code.

`before = len(df)` — we save the current number of rows first.
This is like taking a headcount before removing anyone from the room.
We need this number later to calculate how many rows were removed.

`df['year'] >= 1980` — this checks every row and asks "is this year 1980 or later?"
It returns True for rows that pass and False for rows that fail.
Think of it like a yes/no question asked to every single row.

`df['year'] <= 2026` — same idea but asking "is this year 2026 or earlier?"

`&` — this means AND. Both conditions must be True at the same time.
A row only survives if the year is >= 1980 AND <= 2026.
If either condition is False, the row is removed.

`df = df[...]` — we overwrite df with only the rows that passed both conditions.
The rows that failed simply disappear from our dataset.

`before - len(df)` — subtracting the new count from the old count
tells us exactly how many rows were removed in this step.

## 8. Handling Missing Values

Now we deal with rows that have empty values in important columns.
We cannot build a price prediction model if the price itself is missing.
We also cannot use a row if the year, mileage, or engine is missing.

Here is our rule:
- Drop rows where price, year, mileage, or engine could not be cleaned
- These are the four most important columns for predicting price

In [ ]:
before = len(df)
df = df.dropna(subset=['price_pkr', 'year', 'mileage_km', 'engine_cc'])
print(f"Rows before: {before}")
print(f"Rows after:  {len(df)}")
print(f"Removed:     {before - len(df)}")

Let us break down every single part of this code.

`before = len(df)` — we save the row count before dropping anything.
Same trick as before — we always save the count first so we can
calculate how many rows were removed at the end.

`df.dropna(...)` — dropna means "drop rows with NA values".
NA means Not Available — it is just another way of saying missing or empty.
Think of it like removing incomplete application forms.
If a form is missing the name or date, you throw it away.

`subset=['price_pkr', 'year', 'mileage_km', 'engine_cc']` — subset means
"only check these specific columns for missing values".
We do not drop a row just because the city or seller_type is missing.
We only drop it if the four most important columns are missing.

`df = df.dropna(...)` — we overwrite df with the result.
All rows with missing values in those four columns are now gone.

After this step our dataset only contains rows where we know
the price, year, mileage, and engine size. These are the four
columns our model will rely on the most.

## 9. Removing Duplicate Rows

Sometimes the same car listing appears more than once in scraped data.
This happens when the scraper runs multiple times or the same ad
is listed under different categories on PakWheels.

Duplicate rows are a problem because they make our model think
one car is more common than it actually is.
We remove them to keep only unique listings.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['ad_id'])
print(f"Rows before: {before}")
print(f"Rows after:  {len(df)}")
print(f"Removed:     {before - len(df)}")

Let us break down every single part of this code.

`df.drop_duplicates(...)` — this removes rows that are exact copies of each other.
Think of it like having the same student registered twice in a class.
You keep one and remove the other.

`subset=['ad_id']` — ad_id is the unique identifier for each car listing on PakWheels.
Every listing has its own ad_id like a national ID card number.
No two different listings should have the same ad_id.

By checking only the ad_id column, we are saying:
"if two rows have the exact same ad_id, they are the same listing — keep only one."

`df = df.drop_duplicates(...)` — we overwrite df with the cleaned result.
All duplicate listings are now removed.

`before - len(df)` — tells us how many duplicate rows were found and removed.
If this number is 0, it means the data had no duplicates at all.
If it is large, it means the scraper collected a lot of repeated listings.

## 10. Selecting the Final Columns

Right now our DataFrame has 18 original columns plus the 3 new ones we created.
Many of the original columns are either empty, redundant, or not useful for analysis.

We keep only the columns that matter for price prediction and analysis.
This makes the cleaned dataset smaller, cleaner, and easier to work with.

In [ ]:
columns_to_keep = [
    'make', 'year', 'price_pkr', 'mileage_km', 'engine_cc',
    'transmission', 'fuel_type', 'city', 'seller_type'
]
df_clean = df[columns_to_keep].copy()
print(df_clean.shape)
print(df_clean.head(3))

Let us break down every single part of this code.

`columns_to_keep = [...]` — this is a list of column names we want to keep.
Think of it like packing for a trip — you only take what you actually need.
We leave behind columns like url, ad_id, source_url, and scraped_at
because they are not useful for predicting car prices.

Here is why we kept each column:
- make — the brand of the car (Honda, Toyota, Suzuki)
- year — the model year, affects price a lot
- price_pkr — this is what we are trying to predict
- mileage_km — how much the car has been driven
- engine_cc — the engine size in cubic centimetres
- transmission — Automatic or Manual
- fuel_type — Petrol, Diesel, Hybrid, etc.
- city — where the car is being sold
- seller_type — Individual or Dealer

`df[columns_to_keep]` — selects only those columns from df.
All other columns are dropped.

`.copy()` — makes a fresh independent copy of the data.
This is important because without .copy(), changes to df_clean
might accidentally affect df as well.
We always use .copy() when creating a new cleaned version.

## 11. Final Check Before Saving

Before we save the cleaned data, we do one last check.
We want to make sure everything looks correct —
no missing values in important columns, correct data types,
and sensible values throughout.

In [ ]:
print("Shape:", df_clean.shape)
print("\nMissing values:")
print(df_clean.isnull().sum())
print("\nData types:")
print(df_clean.dtypes)

Let us read this output carefully before saving.

Shape tells us how many rows and columns are in the final clean dataset.
Compare this to the original 18 columns — we now have only 9 clean ones.

Missing values section shows how many empty cells remain in each column.
Ideally price_pkr, year, mileage_km, and engine_cc should all show 0.
If any of these show a number above 0, something went wrong in a previous step.

Data types section shows what kind of data each column holds.
- float64 means decimal numbers like 3875000.0
- int64 means whole numbers like 2020
- object means text like "Honda" or "Automatic"

price_pkr, mileage_km, and engine_cc should all be numbers, not object.
If any of them still shows object, the cleaning function did not work properly.

## 12. Saving the Cleaned Data

Everything looks good. Now we save the cleaned dataset.
This file will be used by every notebook that comes after this one.
We never touch the raw file again from this point forward.

In [ ]:
output_path = os.path.join('..', 'data', 'processed', '02_cleaned.csv')
df_clean.to_csv(output_path, index=False)
print(f"Cleaned data saved: {len(df_clean)} rows, {df_clean.shape[1]} columns")

## 13. Cleaning Summary

Here is a complete record of everything this notebook did.
This is good practice — always document what changed and why.

In [ ]:
print("=== Cleaning Summary ===")
print(f"Raw rows loaded:        {len(pd.read_csv(os.path.join('..', 'data', 'raw', 'pakwheels.csv')))}")
print(f"Final clean rows:       {len(df_clean)}")
print(f"Columns kept:           {df_clean.shape[1]} out of 18")
print(f"Price range (PKR):      {df_clean['price_pkr'].min():,.0f} — {df_clean['price_pkr'].max():,.0f}")
print(f"Year range:             {int(df_clean['year'].min())} — {int(df_clean['year'].max())}")
print(f"Unique car makes:       {df_clean['make'].nunique()}")
print(f"Unique cities:          {df_clean['city'].nunique()}")

 *This summary tells the complete story of our cleaning process.*

We started with the raw messy data and ended with a clean dataset
that is ready for analysis and machine learning.

Here is a quick recap of every fix we made:

| Step | What we fixed |
|------|--------------|
| Price | Converted "PKR 38.75 lacs" to 3,875,000 |
| Mileage | Converted "19,198 km" to 19198 |
| Engine | Converted "660 cc" to 660 |
| Make | Extracted brand name from title column |
| Year | Removed cars outside 1980 to 2026 |
| Nulls | Dropped rows missing price, year, mileage, or engine |
| Duplicates | Removed repeated listings using ad_id |
| Columns | Kept 9 most useful columns out of 18 |

The next notebook, 03_eda, will explore this clean data
and uncover patterns and insights about the Pakistani used car market.

---

## Quick Reference — Methods Used in This Notebook

| Code | What it does |
|------|-------------|
| `re.search(r'[\d.]+', text)` | Finds numbers inside messy text |
| `number.group()` | Grabs the matched text from re.search |
| `float(...)` | Converts text to a decimal number |
| `int(...)` | Converts text to a whole number |
| `str.replace(',', '')` | Removes commas from text |
| `str.split().str[0]` | Grabs the first word from text |
| `str.strip()` | Removes hidden spaces from text |
| `df.apply(func)` | Runs a function on every row |
| `df[(condition)]` | Keeps only rows where condition is True |
| `df.dropna(subset=[...])` | Drops rows with missing values in specific columns |
| `df.drop_duplicates(subset=[...])` | Removes duplicate rows based on a column |
| `df[columns].copy()` | Selects specific columns and makes a clean copy |